In [6]:
import pandas as pd
import panel as pn
import hvplot.pandas

pn.extension('tabulator')

# -------------------------------
# LOAD DATA
# -------------------------------
df = pd.read_csv("AQI_RAW.csv")

# CLEANING
df.columns = df.columns.str.strip().str.lower()
df['last_update'] = pd.to_datetime(df['last_update'], format='%d-%m-%Y %H:%M')
df['pollutant_avg'] = df['pollutant_avg'].fillna(df['pollutant_avg'].mean())
df = df.drop_duplicates()

In [7]:
df.head()

,country,state,city,station,last_update,latitude,longitude,pollutant_id,pollutant_min,pollutant_max,pollutant_avg
0,India,Assam,Guwahati,"Railway Colony, Guwahati - PCBA",2026-04-05 08:00:00,26.181742,91.780630,SO2,1.0,8.0,4.000000
1,India,Assam,Nagaon,"Christianpatty, Nagaon - PCBA",2026-04-05 08:00:00,26.349082,92.684490,NH3,3.0,4.0,4.000000
2,India,Assam,Nagaon,"Christianpatty, Nagaon - PCBA",2026-04-05 08:00:00,26.349082,92.684490,SO2,6.0,15.0,12.000000
3,India,Assam,Nalbari,"Bata Chowk, Nalbari - PCBA",2026-04-05 08:00:00,26.446912,91.439057,PM10,NaN,NaN,34.907755
4,India,Assam,Silchar,"Tarapur, Silchar - PCBA",2026-04-05 08:00:00,24.828270,92.795250,NO2,24.0,31.0,28.000000


In [8]:
# PIVOT
df_pivot = df.pivot_table(
    index=['city', 'last_update'],
    columns='pollutant_id',
    values='pollutant_avg'
).reset_index()

df_pivot.columns.name = None

In [9]:
df_pivot.head()

,city,last_update,CO,NH3,NO2,OZONE,PM10,PM2.5,SO2
0,Agartala,2026-04-05 08:00:00,20.000,7.000000,20.00,2.000000,96.000000,82.000000,4.000000
1,Agra,2026-04-05 08:00:00,14.000,5.200000,22.50,21.302585,75.333333,41.333333,22.333333
2,Ahmedabad,2026-04-05 08:00:00,51.125,6.285714,54.75,17.571429,100.125000,59.875000,34.000000
3,Ahmednagar,2026-04-05 08:00:00,74.000,6.000000,21.00,14.000000,61.000000,40.000000,8.000000
4,Aizawl,2026-04-05 08:00:00,3.000,NaN,1.00,NaN,91.000000,74.000000,15.000000


In [10]:
# AQI
pollutants = ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3', 'NH3']
for p in pollutants:
    if p not in df_pivot.columns:
        df_pivot[p] = 0

df_pivot['aqi'] = df_pivot[pollutants].max(axis=1)

# CATEGORY
def categorize_aqi(aqi):
    if aqi <= 50: return "Good"
    elif aqi <= 100: return "Satisfactory"
    elif aqi <= 200: return "Moderate"
    elif aqi <= 300: return "Poor"
    elif aqi <= 400: return "Very Poor"
    else: return "Severe"

df_pivot['aqi_category'] = df_pivot['aqi'].apply(categorize_aqi)

# -------------------------------
# KPI CARDS 
# -------------------------------
avg_aqi = int(df_pivot['aqi'].mean())
max_aqi = int(df_pivot['aqi'].max())
worst_city = df_pivot.loc[df_pivot['aqi'].idxmax(), 'city']

kpi1 = pn.pane.Markdown(f"###  Avg AQI\n# {avg_aqi}")
kpi2 = pn.pane.Markdown(f"###  Max AQI\n# {max_aqi}")
kpi3 = pn.pane.Markdown(f"### Worst City\n# {worst_city}")

# -------------------------------
# VISUALS
# -------------------------------

# AQI Category
aqi_bar = df_pivot['aqi_category'].value_counts().hvplot.bar(
    title="AQI Category Distribution",
    height=300
)

# Top Cities
top_city_plot = df_pivot.groupby('city')['aqi'].mean().sort_values(ascending=False).head(10).hvplot.bar(
    title="Top 10 Most Polluted Cities",
    height=300
)

# Scatter
scatter = df_pivot.hvplot.scatter(
    x='PM2.5', y='aqi',
    title="PM2.5 vs AQI",
    size=60,
    height=300
)

# Pollutants
pollutant_plot = df_pivot[pollutants].mean().hvplot.bar(
    title="Average Pollutant Levels",
    height=300
)

# Table
aqi_table = df_pivot[['city', 'aqi', 'aqi_category']].sort_values(
    by='aqi', ascending=False
).head(20)

table_panel = pn.widgets.Tabulator(aqi_table)

# -------------------------------
# TEMPLATE (FILLED LAYOUT)
# -------------------------------
template = pn.template.FastListTemplate(
    title='India AQI Dashboard',

    sidebar=[
        pn.pane.Markdown("""
# Air Quality Index Dashboard

### What is AQI?
Air Quality Index (AQI) is a measure used to describe how polluted the air is.  
Higher AQI values indicate worse air quality and greater health risks.

---

### Health Impact Levels
- Good (0–50): Clean air, minimal impact  
- Satisfactory (51–100): Minor breathing discomfort  
- Moderate (101–200): Discomfort for sensitive groups  
- Poor (201–300): Breathing discomfort for most people  
- Very Poor (301–400): Increased risk of respiratory illness  
- Severe (401+): Serious health effects  

---

### Pollutants Considered
- PM2.5  
- PM10  
- NO2  
- SO2  
- CO  
- O3  
- NH3  

---

### Methodology
- Data cleaned using Python (Pandas)  
- Transformed from long format to wide format using pivot operation  
- AQI calculated using the maximum pollutant concentration  
- Visualizations created using Panel and hvPlot  

---

### Key Insight
PM2.5 has the strongest influence on AQI values across most cities.

---

### Tools Used
- Python  
- Pandas  
- Panel  
- hvPlot  
""", sizing_mode='stretch_width')
    ],

    main=[
        # KPI ROW
        pn.Row(kpi1, kpi2, kpi3),

        # MAIN VISUALS
        pn.Row(aqi_bar, top_city_plot),

        pn.Row(scatter, pollutant_plot),

        pn.Row(table_panel)
    ],

    accent_base_color="#ff6f69",
    header_background="#ff6f69",
)

template.show()
template.servable()

Launching server at http://localhost:4586


FastListTemplate
    [js_area] HTML(None, height=0, margin=0, sizing_mode='fixed', width=0)
    [actions] TemplateActions()
    [browser_info] BrowserInfo(dark_mode=True, device_pixel_ratio=1.600000023841858, language='en-US', timezone='Asia/Calcutta', timezone_offset=-330, webdriver=False, webgl=True)
    [busy_indicator] LoadingSpinner(height=20, width=20)
    [main-2274584737872] Row
        [0] Markdown(str)
        [1] Markdown(str)
        [2] Markdown(str)
    [main-2274584738320] Row
        [0] HoloViews(Bars, height=300, sizing_mode='fixed', width=700)
        [1] HoloViews(Bars, height=300, sizing_mode='fixed', width=700)
    [main-2274602175632] Row
        [0] HoloViews(Scatter, height=300, sizing_mode='fixed', width=700)
        [1] HoloViews(Bars, height=300, sizing_mode='fixed', width=700)
    [main-2274602180112] Row
        [0] Tabulator(value=              ...)
    [nav-2274584200880] Markdown(str, sizing_mode='stretch_width')